# Pass/Fail Prediction - Data Preparation

This notebook prepares the UCI Student Performance and xAPI-Edu-Data datasets for pass/fail prediction with aligned features.

## Objectives:
1. Load UCI and xAPI datasets
2. Align features between datasets (create common feature set)
3. Create pass/fail target variables
4. Normalize features
5. Save prepared datasets ready for train/test split and model training


In [1]:
# Install required packages
!pip install -q pandas numpy scikit-learn



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
import os
import warnings
warnings.filterwarnings('ignore')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")


Libraries imported successfully!


## 1. Load Datasets


In [3]:
# Find dataset paths
# From pass-fail-prediction/ we need to go up 4 levels to reach project root
base_paths_uci = [
    '../../../../datasets/uci-student-performance/',  # From pass-fail-prediction/ (4 levels up)
    '../../../datasets/uci-student-performance/',    # From analysis/ (3 levels up)
    '../../datasets/uci-student-performance/',        # From charaka/ (2 levels up)
    'datasets/uci-student-performance/'               # From project root
]

base_paths_xapi = [
    '../../../../datasets/xapi-edu-data/xAPI-Edu-Data.csv',  # From pass-fail-prediction/ (4 levels up)
    '../../../datasets/xapi-edu-data/xAPI-Edu-Data.csv',    # From analysis/ (3 levels up)
    '../../datasets/xapi-edu-data/xAPI-Edu-Data.csv',        # From charaka/ (2 levels up)
    'datasets/xapi-edu-data/xAPI-Edu-Data.csv'              # From project root
]

# Find UCI dataset directory
uci_dir = None
for path in base_paths_uci:
    if os.path.exists(path):
        uci_dir = path
        break

if uci_dir is None:
    raise FileNotFoundError(f"Could not find UCI dataset directory. Tried: {base_paths_uci}")

# Find xAPI dataset file
xapi_path = None
for path in base_paths_xapi:
    if os.path.exists(path):
        xapi_path = path
        break

if xapi_path is None:
    raise FileNotFoundError(f"Could not find xAPI dataset. Tried: {base_paths_xapi}")

print(f"UCI dataset directory: {os.path.abspath(uci_dir)}")
print(f"xAPI dataset path: {os.path.abspath(xapi_path)}")


UCI dataset directory: /Users/charaka/Desktop/Projects/uom-student-performance-analytics/datasets/uci-student-performance
xAPI dataset path: /Users/charaka/Desktop/Projects/uom-student-performance-analytics/datasets/xapi-edu-data/xAPI-Edu-Data.csv


In [4]:
# Load UCI datasets (semicolon-separated)
df_uci_math = pd.read_csv(os.path.join(uci_dir, 'student-mat.csv'), sep=';')
df_uci_por = pd.read_csv(os.path.join(uci_dir, 'student-por.csv'), sep=';')

# Combine UCI datasets (or use separately if needed)
# Note: 382 students appear in both, but we'll combine for more data
df_uci = pd.concat([df_uci_math, df_uci_por], ignore_index=True)

# Load xAPI dataset
df_xapi = pd.read_csv(xapi_path)

# Fix xAPI column name capitalization issues
df_xapi.rename(columns={'VisITedResources': 'VisitedResources', 'NationalITy': 'Nationality'}, inplace=True)

print(f"UCI Math dataset: {df_uci_math.shape[0]} rows × {df_uci_math.shape[1]} columns")
print(f"UCI Portuguese dataset: {df_uci_por.shape[0]} rows × {df_uci_por.shape[1]} columns")
print(f"UCI Combined dataset: {df_uci.shape[0]} rows × {df_uci.shape[1]} columns")
print(f"xAPI dataset: {df_xapi.shape[0]} rows × {df_xapi.shape[1]} columns")


UCI Math dataset: 395 rows × 33 columns
UCI Portuguese dataset: 649 rows × 33 columns
UCI Combined dataset: 1044 rows × 33 columns
xAPI dataset: 480 rows × 17 columns


## 2. Create Pass/Fail Target Variables


In [5]:
# UCI: Create pass/fail from G3 (final grade)
# Threshold: G3 >= 10 = Pass (1), G3 < 10 = Fail (0)
df_uci['pass_fail'] = (df_uci['G3'] >= 10).astype(int)
df_uci['pass_fail_label'] = df_uci['pass_fail'].map({1: 'Pass', 0: 'Fail'})

# xAPI: Create pass/fail from Class
# L = Low = Fail (0), M/H = Medium/High = Pass (1)
df_xapi['pass_fail'] = (df_xapi['Class'].isin(['M', 'H'])).astype(int)
df_xapi['pass_fail_label'] = df_xapi['pass_fail'].map({1: 'Pass', 0: 'Fail'})

print("UCI Pass/Fail distribution:")
print(df_uci['pass_fail_label'].value_counts())
print(f"\nPass rate: {df_uci['pass_fail'].mean():.2%}")

print("\nxAPI Pass/Fail distribution:")
print(df_xapi['pass_fail_label'].value_counts())
print(f"\nPass rate: {df_xapi['pass_fail'].mean():.2%}")


UCI Pass/Fail distribution:
pass_fail_label
Pass    814
Fail    230
Name: count, dtype: int64

Pass rate: 77.97%

xAPI Pass/Fail distribution:
pass_fail_label
Pass    353
Fail    127
Name: count, dtype: int64

Pass rate: 73.54%


## 3. Feature Engineering and Alignment


In [6]:
# ========== UCI Dataset Feature Engineering ==========

# Create a copy for processing
df_uci_processed = df_uci.copy()

# 1. Gender: sex -> gender (standardize naming)
df_uci_processed['gender'] = df_uci_processed['sex'].map({'M': 'M', 'F': 'F'})

# 2. Study time: already numeric (1-4), normalize later
# Keep as is for now

# 3. Absences: already numeric (0-93), normalize later
# Keep as is for now

# 4. Failures: already numeric, normalize later
# Keep as is for now

# 5. Family support: famsup (yes/no) -> binary (1/0)
df_uci_processed['famsup_binary'] = (df_uci_processed['famsup'] == 'yes').astype(int)

print("UCI dataset processed.")
print(f"Shape: {df_uci_processed.shape}")


UCI dataset processed.
Shape: (1044, 37)


In [7]:
# ========== xAPI Dataset Feature Engineering ==========

# Create a copy for processing
df_xapi_processed = df_xapi.copy()

# 1. Gender: already named 'gender', keep as is
# Standardize values if needed
if df_xapi_processed['gender'].dtype == 'object':
    df_xapi_processed['gender'] = df_xapi_processed['gender'].str.upper().str[0]  # M/F

# 2. Absences: StudentAbsenceDays (categorical) -> numeric
# Under-7 -> 3.5 (midpoint), Above-7 -> 10 (estimate)
df_xapi_processed['absences_numeric'] = df_xapi_processed['StudentAbsenceDays'].map({
    'Under-7': 3.5,
    'Above-7': 10.0
})

# 3. Engagement Score: composite from behavioral metrics
# Normalize each metric first, then combine
behavioral_cols = ['raisedhands', 'VisitedResources', 'AnnouncementsView', 'Discussion']

# Check if columns exist
for col in behavioral_cols:
    if col not in df_xapi_processed.columns:
        print(f"Warning: Column {col} not found. Available columns: {df_xapi_processed.columns.tolist()}")

# Normalize behavioral metrics to 0-1 scale
for col in behavioral_cols:
    if col in df_xapi_processed.columns:
        col_min = df_xapi_processed[col].min()
        col_max = df_xapi_processed[col].max()
        if col_max > col_min:
            df_xapi_processed[f"{col}_normalized"] = (df_xapi_processed[col] - col_min) / (col_max - col_min)
        else:
            df_xapi_processed[f"{col}_normalized"] = 0.0

# Create engagement score: weighted combination
if all(f"{col}_normalized" in df_xapi_processed.columns for col in behavioral_cols):
    df_xapi_processed['engagement_score'] = (
        df_xapi_processed['raisedhands_normalized'] * 0.3 +
        df_xapi_processed['VisitedResources_normalized'] * 0.3 +
        df_xapi_processed['AnnouncementsView_normalized'] * 0.2 +
        df_xapi_processed['Discussion_normalized'] * 0.2
    )
else:
    print("Warning: Could not create engagement_score. Using available metrics.")
    # Fallback: use average of available normalized metrics
    available_norm = [f"{col}_normalized" for col in behavioral_cols if f"{col}_normalized" in df_xapi_processed.columns]
    if available_norm:
        df_xapi_processed['engagement_score'] = df_xapi_processed[available_norm].mean(axis=1)
    else:
        df_xapi_processed['engagement_score'] = 0.0

# 4. Parental Support: derived from surveys
df_xapi_processed['parental_support'] = (
    (df_xapi_processed['ParentAnsweringSurvey'] == 'Yes') &
    (df_xapi_processed['ParentschoolSatisfaction'] == 'Good')
).astype(int)

# 5. Academic Level: derive from GradeID/StageID
# Map to numeric scale (higher grade/stage = higher academic level)
if 'GradeID' in df_xapi_processed.columns:
    # Create numeric mapping for GradeID
    grade_mapping = {}
    unique_grades = sorted(df_xapi_processed['GradeID'].unique())
    for idx, grade in enumerate(unique_grades):
        grade_mapping[grade] = idx + 1
    df_xapi_processed['academic_level'] = df_xapi_processed['GradeID'].map(grade_mapping)
elif 'StageID' in df_xapi_processed.columns:
    # Use StageID as fallback
    stage_mapping = {}
    unique_stages = sorted(df_xapi_processed['StageID'].unique())
    for idx, stage in enumerate(unique_stages):
        stage_mapping[stage] = idx + 1
    df_xapi_processed['academic_level'] = df_xapi_processed['StageID'].map(stage_mapping)
else:
    # Default: set to 1 if no grade/stage info
    df_xapi_processed['academic_level'] = 1

print("xAPI dataset processed.")
print(f"Shape: {df_xapi_processed.shape}")


xAPI dataset processed.
Shape: (480, 27)


## 4. Create Common Feature Set


In [8]:
# Define common feature set
# These features will be present in both datasets after alignment

common_features = {
    'gender': 'gender',  # Direct match
    'absences': 'absences_numeric',  # UCI: absences, xAPI: absences_numeric
    'engagement': 'engagement_score',  # UCI: studytime, xAPI: engagement_score
    'parental_support': 'parental_support',  # UCI: famsup_binary, xAPI: parental_support
    'academic_level': 'academic_level',  # UCI: failures, xAPI: academic_level
}

# UCI feature mapping
uci_feature_map = {
    'gender': 'gender',
    'absences': 'absences',
    'engagement': 'studytime',  # Will normalize later
    'parental_support': 'famsup_binary',
    'academic_level': 'failures',  # Will normalize later
}

# xAPI feature mapping
xapi_feature_map = {
    'gender': 'gender',
    'absences': 'absences_numeric',
    'engagement': 'engagement_score',  # Already normalized 0-1
    'parental_support': 'parental_support',
    'academic_level': 'academic_level',
}

print("Common feature set defined:")
for common_name, description in common_features.items():
    print(f"  - {common_name}: {description}")


Common feature set defined:
  - gender: gender
  - absences: absences_numeric
  - engagement: engagement_score
  - parental_support: parental_support
  - academic_level: academic_level


In [9]:
# Extract common features for UCI
df_uci_common = pd.DataFrame()
df_uci_common['gender'] = df_uci_processed[uci_feature_map['gender']]
df_uci_common['absences'] = df_uci_processed[uci_feature_map['absences']]
df_uci_common['engagement'] = df_uci_processed[uci_feature_map['engagement']]
df_uci_common['parental_support'] = df_uci_processed[uci_feature_map['parental_support']]
df_uci_common['academic_level'] = df_uci_processed[uci_feature_map['academic_level']]
df_uci_common['pass_fail'] = df_uci_processed['pass_fail']
df_uci_common['pass_fail_label'] = df_uci_processed['pass_fail_label']

# Extract common features for xAPI
df_xapi_common = pd.DataFrame()
df_xapi_common['gender'] = df_xapi_processed[xapi_feature_map['gender']]
df_xapi_common['absences'] = df_xapi_processed[xapi_feature_map['absences']]
df_xapi_common['engagement'] = df_xapi_processed[xapi_feature_map['engagement']]
df_xapi_common['parental_support'] = df_xapi_processed[xapi_feature_map['parental_support']]
df_xapi_common['academic_level'] = df_xapi_processed[xapi_feature_map['academic_level']]
df_xapi_common['pass_fail'] = df_xapi_processed['pass_fail']
df_xapi_common['pass_fail_label'] = df_xapi_processed['pass_fail_label']

print("UCI common features shape:", df_uci_common.shape)
print("xAPI common features shape:", df_xapi_common.shape)
print("\nUCI common features columns:", df_uci_common.columns.tolist())
print("xAPI common features columns:", df_xapi_common.columns.tolist())


UCI common features shape: (1044, 7)
xAPI common features shape: (480, 7)

UCI common features columns: ['gender', 'absences', 'engagement', 'parental_support', 'academic_level', 'pass_fail', 'pass_fail_label']
xAPI common features columns: ['gender', 'absences', 'engagement', 'parental_support', 'academic_level', 'pass_fail', 'pass_fail_label']


## 5. Encode Categorical Features


In [10]:
# Encode gender (M/F -> 0/1)
gender_encoder = LabelEncoder()

# Fit on combined data to ensure consistent encoding
all_genders = pd.concat([df_uci_common['gender'], df_xapi_common['gender']])
gender_encoder.fit(all_genders)

# Transform both datasets
df_uci_common['gender_encoded'] = gender_encoder.transform(df_uci_common['gender'])
df_xapi_common['gender_encoded'] = gender_encoder.transform(df_xapi_common['gender'])

print("Gender encoding:")
print(gender_encoder.classes_)
print(f"\nUCI gender distribution:")
print(df_uci_common['gender'].value_counts())
print(f"\nxAPI gender distribution:")
print(df_xapi_common['gender'].value_counts())


Gender encoding:
['F' 'M']

UCI gender distribution:
gender
F    591
M    453
Name: count, dtype: int64

xAPI gender distribution:
gender
M    305
F    175
Name: count, dtype: int64


## 6. Normalize Numeric Features


In [11]:
# Features to normalize
numeric_features = ['absences', 'engagement', 'academic_level']

# Combine datasets for fitting scalers (to ensure same scale)
combined_numeric = pd.concat([
    df_uci_common[numeric_features],
    df_xapi_common[numeric_features]
], ignore_index=True)

# Fit scalers on combined data
scalers = {}
for feature in numeric_features:
    scaler = MinMaxScaler()  # Normalize to 0-1 range
    scaler.fit(combined_numeric[[feature]])
    scalers[feature] = scaler

# Transform UCI dataset
for feature in numeric_features:
    df_uci_common[f"{feature}_normalized"] = scalers[feature].transform(df_uci_common[[feature]])

# Transform xAPI dataset
for feature in numeric_features:
    df_xapi_common[f"{feature}_normalized"] = scalers[feature].transform(df_xapi_common[[feature]])

print("Features normalized successfully.")
print("\nNormalized feature ranges (UCI):")
for feature in numeric_features:
    print(f"  {feature}_normalized: [{df_uci_common[f'{feature}_normalized'].min():.3f}, {df_uci_common[f'{feature}_normalized'].max():.3f}]")

print("\nNormalized feature ranges (xAPI):")
for feature in numeric_features:
    print(f"  {feature}_normalized: [{df_xapi_common[f'{feature}_normalized'].min():.3f}, {df_xapi_common[f'{feature}_normalized'].max():.3f}]")


Features normalized successfully.

Normalized feature ranges (UCI):
  absences_normalized: [0.000, 1.000]
  engagement_normalized: [0.249, 1.000]
  academic_level_normalized: [0.000, 0.300]

Normalized feature ranges (xAPI):
  absences_normalized: [0.047, 0.133]
  engagement_normalized: [0.000, 0.234]
  academic_level_normalized: [0.100, 1.000]


## 7. Create Final Prepared Datasets


In [12]:
# Create final feature sets (all normalized/encoded)
# Use normalized versions and encoded gender

feature_columns = [
    'gender_encoded',
    'absences_normalized',
    'engagement_normalized',
    'parental_support',  # Already binary 0/1
    'academic_level_normalized'
]

# UCI final dataset
df_uci_final = pd.DataFrame()
for col in feature_columns:
    df_uci_final[col] = df_uci_common[col]
df_uci_final['pass_fail'] = df_uci_common['pass_fail']
df_uci_final['pass_fail_label'] = df_uci_common['pass_fail_label']

# xAPI final dataset
df_xapi_final = pd.DataFrame()
for col in feature_columns:
    df_xapi_final[col] = df_xapi_common[col]
df_xapi_final['pass_fail'] = df_xapi_common['pass_fail']
df_xapi_final['pass_fail_label'] = df_xapi_common['pass_fail_label']

print("Final prepared datasets:")
print(f"\nUCI final shape: {df_uci_final.shape}")
print(f"UCI columns: {df_uci_final.columns.tolist()}")
print(f"\nxAPI final shape: {df_xapi_final.shape}")
print(f"xAPI columns: {df_xapi_final.columns.tolist()}")

# Check for missing values
print("\nMissing values:")
print(f"UCI: {df_uci_final.isnull().sum().sum()} missing values")
print(f"xAPI: {df_xapi_final.isnull().sum().sum()} missing values")

# Display sample
print("\nUCI sample (first 5 rows):")
print(df_uci_final.head())
print("\nxAPI sample (first 5 rows):")
print(df_xapi_final.head())


Final prepared datasets:

UCI final shape: (1044, 7)
UCI columns: ['gender_encoded', 'absences_normalized', 'engagement_normalized', 'parental_support', 'academic_level_normalized', 'pass_fail', 'pass_fail_label']

xAPI final shape: (480, 7)
xAPI columns: ['gender_encoded', 'absences_normalized', 'engagement_normalized', 'parental_support', 'academic_level_normalized', 'pass_fail', 'pass_fail_label']

Missing values:
UCI: 0 missing values
xAPI: 0 missing values

UCI sample (first 5 rows):
   gender_encoded  absences_normalized  engagement_normalized  \
0               0             0.080000               0.499234   
1               0             0.053333               0.499234   
2               0             0.133333               0.499234   
3               0             0.026667               0.749617   
4               0             0.053333               0.499234   

   parental_support  academic_level_normalized  pass_fail pass_fail_label  
0                 0                    

## 8. Save Prepared Datasets


In [13]:
# Create output directory
output_dir = 'prepared_data'
os.makedirs(output_dir, exist_ok=True)

# Save prepared datasets
uci_output_path = os.path.join(output_dir, 'uci_pass_fail_prepared.csv')
xapi_output_path = os.path.join(output_dir, 'xapi_pass_fail_prepared.csv')

df_uci_final.to_csv(uci_output_path, index=False)
df_xapi_final.to_csv(xapi_output_path, index=False)

print(f"Datasets saved successfully!")
print(f"UCI dataset: {os.path.abspath(uci_output_path)}")
print(f"xAPI dataset: {os.path.abspath(xapi_output_path)}")

# Save feature information
feature_info = {
    'feature_columns': feature_columns,
    'target_column': 'pass_fail',
    'target_label_column': 'pass_fail_label',
    'uci_samples': len(df_uci_final),
    'xapi_samples': len(df_xapi_final),
    'feature_descriptions': {
        'gender_encoded': 'Gender (0=F, 1=M)',
        'absences_normalized': 'Absences normalized to 0-1',
        'engagement_normalized': 'Engagement/study time normalized to 0-1',
        'parental_support': 'Parental support (0=No, 1=Yes)',
        'academic_level_normalized': 'Academic level/failures normalized to 0-1'
    }
}

import json
info_path = os.path.join(output_dir, 'feature_info.json')
with open(info_path, 'w') as f:
    json.dump(feature_info, f, indent=2)

print(f"\nFeature information saved: {os.path.abspath(info_path)}")


Datasets saved successfully!
UCI dataset: /Users/charaka/Desktop/Projects/uom-student-performance-analytics/pretests/charaka/analysis/pass-fail-prediction/prepared_data/uci_pass_fail_prepared.csv
xAPI dataset: /Users/charaka/Desktop/Projects/uom-student-performance-analytics/pretests/charaka/analysis/pass-fail-prediction/prepared_data/xapi_pass_fail_prepared.csv

Feature information saved: /Users/charaka/Desktop/Projects/uom-student-performance-analytics/pretests/charaka/analysis/pass-fail-prediction/prepared_data/feature_info.json


## 9. Summary Statistics


In [14]:
print("=" * 60)
print("DATA PREPARATION SUMMARY")
print("=" * 60)

print("\nDataset Sizes:")
print(f"  UCI: {len(df_uci_final)} samples")
print(f"  xAPI: {len(df_xapi_final)} samples")

print("\nFeature Set:")
for i, feature in enumerate(feature_columns, 1):
    print(f"  {i}. {feature}")

print("\nTarget Distribution (UCI):")
print(df_uci_final['pass_fail_label'].value_counts())
print(f"  Pass rate: {df_uci_final['pass_fail'].mean():.2%}")

print("\nTarget Distribution (xAPI):")
print(df_xapi_final['pass_fail_label'].value_counts())
print(f"  Pass rate: {df_xapi_final['pass_fail'].mean():.2%}")

print("\nFeature Statistics (UCI):")
print(df_uci_final[feature_columns].describe())

print("\nFeature Statistics (xAPI):")
print(df_xapi_final[feature_columns].describe())

print("\n" + "=" * 60)
print("Datasets are ready for train/test split and model training!")
print("=" * 60)


DATA PREPARATION SUMMARY

Dataset Sizes:
  UCI: 1044 samples
  xAPI: 480 samples

Feature Set:
  1. gender_encoded
  2. absences_normalized
  3. engagement_normalized
  4. parental_support
  5. academic_level_normalized

Target Distribution (UCI):
pass_fail_label
Pass    814
Fail    230
Name: count, dtype: int64
  Pass rate: 77.97%

Target Distribution (xAPI):
pass_fail_label
Pass    353
Fail    127
Name: count, dtype: int64
  Pass rate: 73.54%

Feature Statistics (UCI):
       gender_encoded  absences_normalized  engagement_normalized  \
count     1044.000000          1044.000000            1044.000000   
mean         0.433908             0.059132               0.491799   
std          0.495850             0.082800               0.208908   
min          0.000000             0.000000               0.248850   
25%          0.000000             0.000000               0.248850   
50%          0.000000             0.026667               0.499234   
75%          1.000000             0.08000